# Jev-CeNN · Layer 15: Frozen FFNs, CeNN ablation, W129/W257/W512, efficient kernels, progressive attention

**Scope:** Replace only the attention submodule `encoder.layers[15].attn` (NOT 3-to-1 block compression). The 28 FFNs, native attention LayerNorms and residual paths, and task head remain unmodified and frozen.

**Experiments:** (1) Same-checkpoint CeNN on/off, (2) 129/257/512 attention window with 512 as global baseline for T<=512, (3) progressive attention-only layers 16 and 17 with cumulative rollback gates, (4) dense SDPA vs memory-bounded chunked SDPA vs optional torch.compile FlexAttention; actual backend and fallback reported, (5) paired inference timings on genuinely short/long inputs, batch 1/2.

**Design:** 180 cached updates per layer15 window, 100 per progressive layer, case-separated TRAIN/PROBE/DEV/HOLDOUT; final holdout evaluated once. Quality passing is separate from measurable speed improvements. Original FFNs are preserved byte-identically (SHA-256 audit). No GPU results are presumed.

**Run in order with a CUDA GPU.** FlexAttention compilation may be slow or unsupported; enable/disable in setup. The original notebooks remain unchanged.


In [ ]:
#@title 1. Setup: layer 15, three attention windows, frozen native FFNs
%pip -q install "transformers==4.57.6" "datasets>=2.20,<5" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"

from pathlib import Path
import copy, gc, json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items, normalize_question
)
from tinycenn_lm.decision_training import require_finite
from tinycenn_lm.jev_cenn_full_local_repeat import cpu_state_snapshot
from tinycenn_lm.laya_lab.core import BaseLayaReplacementAttention
from tinycenn_lm.jev_cenn_pairwindow_superlayer import CellularMultiscale
from tinycenn_lm.laya_lab.core import _valid_tokens

SEED=2028
SOURCE_MODEL="convaiinnovations/laya-typed-decisions" #@param {type:"string"}
TOP_LAYER=15 #@param {type:"integer"}
MAX_LEN=512
WINDOWS=(129,257,512)
PROGRESSIVE_LAYERS=(16,17) # change to () to skip progressive layers
PROGRESSIVE_WINDOW=129
TRAIN_STEPS=180 #@param {type:"integer"}
PROGRESSIVE_STEPS=100 #@param {type:"integer"}
CACHE_BATCHES=48 #@param {type:"integer"}
PROBE_BATCHES=12
BATCH=2
PROBE_EVERY=30
LR_ATTENTION=2e-5 #@param {type:"number"}
LR_CENN=3e-4 #@param {type:"number"}
TRAIN_CASES=700
DEV_CASES=220
HOLDOUT_CASES=200
PROBE_CASES=80
DEV_GATE_DECISIONS=240
HOLDOUT_DECISIONS=500
COSINE_WEIGHT=3.0
TARGET_COSINE=0.99
TARGET_NMSE=0.025
TARGET_AGREEMENT=0.99
MAX_GOLD_DROP=0.015
MAX_INCREMENTAL_AGREEMENT_DROP=0.01
USE_FP16_AMP=False
BENCH_REPEATS=16
ENABLE_FLEX=True #@param {type:"boolean"}
FLEX_VALIDATE_ATOL=5e-3
MAX_BENCH_EXAMPLES=2
REQUIRE_FASTER_FOR_ACCEPTANCE=False
OUTPUT_DIR=Path("/content/Jev_CeNN_L15_Windows_CeNN_Progressive")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
assert TOP_LAYER == 15 and WINDOWS == (129,257,512)
assert all(x in range(28) for x in PROGRESSIVE_LAYERS) and len(set(PROGRESSIVE_LAYERS))==len(PROGRESSIVE_LAYERS)
assert TOP_LAYER not in PROGRESSIVE_LAYERS
assert TRAIN_STEPS > 0 and BATCH > 0
assert CACHE_BATCHES>=PROBE_BATCHES
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Colab.")
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda")
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
source_dir=Path(snapshot_download(
    SOURCE_MODEL,allow_patterns=[
      "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"]))
teacher,source_cfg=build_checkpoint_model(source_dir)
student_eval,_=build_checkpoint_model(source_dir)
tokenizer=AutoTokenizer.from_pretrained(str(source_dir/"tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
student_eval.encoder.config.reference_compile=False
assert len(teacher.encoder.layers)==28
# The default last layer 27 is a global attention layer.
print("GPU:",torch.cuda.get_device_name(0))
print("Target layer:",TOP_LAYER,"| windows:",WINDOWS,"| progressive:",PROGRESSIVE_LAYERS)
print("All pretrained FFNs, norms, remaining attention and task head frozen.")
from tinycenn_lm.standalone_decision import full_attention_indices
GLOBAL_LAYERS=full_attention_indices(teacher)
assert TOP_LAYER in GLOBAL_LAYERS
print("Original global layers:",GLOBAL_LAYERS)
assert all(i not in GLOBAL_LAYERS for i in PROGRESSIVE_LAYERS), "Progressive defaults expect original sliding layers"


In [ ]:
#@title 3. Partition dataset by CASE: TRAIN, PROBE, DEV-GATE and sealed HOLDOUT
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

from tinycenn_lm.jev_cenn_selective import (
    verify_case_partition, judge_replacement, plan_active_layers,
)
rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="train"))
rng=random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES+DEV_CASES+HOLDOUT_CASES<=len(rows), (
    f"Required TRAIN+DEV+HOLDOUT cases exceeds {len(rows)} dataset TRAIN cases")
train_rows=rows[:TRAIN_CASES]
dev_rows=rows[TRAIN_CASES:TRAIN_CASES+DEV_CASES]
holdout_rows=rows[TRAIN_CASES+DEV_CASES:TRAIN_CASES+DEV_CASES+HOLDOUT_CASES]
split_info=verify_case_partition(train_rows,dev_rows,holdout_rows)
probe_rows=dev_rows[:PROBE_CASES]
gate_rows=dev_rows[PROBE_CASES:]
verify_case_partition(train_rows,probe_rows,gate_rows)
assert not {str(x["id"]) for x in holdout_rows}&{
    str(x["id"]) for x in dev_rows}
train_items=make_items_from_rows(train_rows,SEED,True)
probe_items=make_items_from_rows(probe_rows,SEED+1,True)
gate_items=make_items_from_rows(gate_rows,SEED+2,True)
# IMPORTANT: holdout_items are NOT constructed yet. The final evaluation
# cell materializes these previously sealed rows after all decisions are made.
assert len(gate_items)>=DEV_GATE_DECISIONS
assert len(probe_items)>=PROBE_BATCHES*BATCH
print("Independent partitions:",split_info,
      "probe cases",len(probe_rows),"gate cases",len(gate_rows))
print("Optimizer/probe/gate decisions:",len(train_items),
      len(probe_items),len(gate_items))

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

# Only case-isolated TRAIN / PROBE / DEV-GATE inputs are available here.
# The independent HOLDOUT remains sealed until the final evaluation cell.
print("TRAIN / PROBE / DEV-GATE decisions:",
      len(train_items), len(probe_items), len(gate_items))
print("Official TEST untouched; independent HOLDOUT not materialized.")


In [ ]:

#@title 3. Cache teacher layer outputs and current-student prefix (per tested layer)
import hashlib
def check_finite(x,name):
    if not torch.isfinite(x.float()).all(): raise FloatingPointError(name)

def to_cuda(row):
    return {k:v.to(device,non_blocking=True) for k,v in row.items()}

def cache_layer(layer_index,items,n_batches,seed):
    chosen=random.Random(seed).sample(items,min(len(items),n_batches*BATCH))
    outputs=[]
    for offset in range(0,len(chosen)-BATCH+1,BATCH):
        chosen_items=chosen[offset:offset+BATCH]
        inp=direct_batch(chosen_items)
        found={};rollout={}
        def pre(mod,args,kwargs):
            found["x"]=args[0].detach()
            found["position_ids"]=kwargs.get("position_ids")
        def post(mod,args,kwargs,result):
            found["target"]=result[0].detach()
        def current(mod,args,kwargs):
            rollout["x"]=args[0].detach()
        h1=teacher.encoder.layers[layer_index].register_forward_pre_hook(pre,with_kwargs=True)
        h2=teacher.encoder.layers[layer_index].register_forward_hook(post,with_kwargs=True)
        try:
            with torch.inference_mode():teacher(**inp)
        finally:
            h1.remove();h2.remove()
        h3=student_eval.encoder.layers[layer_index].register_forward_pre_hook(current,with_kwargs=True)
        try:
            with torch.inference_mode():student_eval(**inp)
        finally:
            h3.remove()
        assert all(k in found for k in ("x","target","position_ids"))
        assert "x" in rollout
        record={
            "x":found["x"],"rollout":rollout["x"],"end_target":found["target"],
            "valid":inp["attention_mask"].bool(),
            "position_ids":found["position_ids"],
        }
        outputs.append({k:(v.detach().float().cpu().contiguous() if v.is_floating_point()
                           else v.detach().cpu().contiguous())
                        for k,v in record.items()})
    return outputs

def new_layer_caches(layer_index):
    train=cache_layer(layer_index,train_items,CACHE_BATCHES,SEED+layer_index*13+1)
    probe=cache_layer(layer_index,probe_items,PROBE_BATCHES,SEED+layer_index*13+2)
    print("Cached teacher and CURRENT student prefix:",layer_index,len(train),len(probe))
    return train,probe

def frozen_fingerprint(model):
    # Tracks actual frozen FFNs and normalization; changing any is a failure.
    sha=hashlib.sha256()
    for i,layer in enumerate(model.encoder.layers):
        for module_name in ("mlp","mlp_norm","attn_norm"):
            module=getattr(layer,module_name)
            for key,val in module.state_dict().items():
                sha.update(f"{i}:{module_name}:{key}".encode())
                sha.update(val.detach().cpu().contiguous().numpy().tobytes())
    return sha.hexdigest()

ALL_FROZEN_INITIAL=frozen_fingerprint(student_eval)
print("Original FFN/normalization SHA-256:",ALL_FROZEN_INITIAL)


In [ ]:
#@title 6. Cosine/NMSE and cached independent DEV-GATE teacher predictions
def feature_loss(output,reference,valid):
    pred=output.float()[valid]
    gold=reference.float()[valid]
    # Source activations are validated during cache construction; the
    # differentiable scalar loss is checked once per update. Avoid CUDA
    # sync in these two per-forward tensor checks.
    nmse=(pred-gold).square().sum()/gold.square().sum().clamp_min(1e-8)
    cosine=F.cosine_similarity(pred,gold,dim=-1).mean()
    # Do not call Python float() inside the 520-step CUDA training loop.
    # Converting GPU tensors to floats forces a device synchronization.
    return nmse+COSINE_WEIGHT*(1-cosine),nmse.detach(),cosine.detach()
@torch.inference_mode()
def cached_local_quality(candidate,cache,key="rollout"):
    candidate.eval()
    cs=err=energy=tokens=0.0
    for row in cache:
        e=to_cuda(row)
        with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            pred=candidate(e[key],attention_mask=e["valid"],
                           position_ids=e["position_ids"])[0]
        a=pred.float()[e["valid"]]
        b=e["end_target"].float()[e["valid"]]
        check_finite(a,"local prediction")
        cs+=float(F.cosine_similarity(a,b,dim=-1).sum())
        err+=float((a-b).square().sum())
        energy+=float(b.square().sum())
        tokens+=int(e["valid"].sum())
    return {"cosine":cs/max(tokens,1),"nmse":err/max(energy,1e-8),
            "tokens":int(tokens)}
def cached_rank(teacher_view,prefix_view):
    return (0.20*(teacher_view["nmse"]+COSINE_WEIGHT*(1-teacher_view["cosine"]))
           +0.80*(prefix_view["nmse"]+COSINE_WEIGHT*(1-prefix_view["cosine"])))
def cache_teacher_predictions(items):
    examples=[]
    for i in range(0,len(items),BATCH):
        inp=direct_batch(items[i:i+BATCH])
        with torch.inference_mode(),torch.autocast(
                "cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            logits,_=teacher(**inp)
        examples.append({
            "batch":inp,
            "teacher":logits.float().detach(),
            "gold":torch.tensor(
                [r["gold_index"] for r in items[i:i+BATCH]],
                device=device,dtype=torch.long)})
    return examples
GATE_ITEMS=gate_items[:DEV_GATE_DECISIONS]
assert len(GATE_ITEMS)==DEV_GATE_DECISIONS
GATE_CACHE=cache_teacher_predictions(GATE_ITEMS)
@torch.inference_mode()
def decision_quality(model,examples):
    model.eval()
    correct=student_gold=teacher_gold=n=0
    js_sum=0.
    for e in examples:
        with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            logits,_=model(**e["batch"])
        check_finite(logits,"full-model decision logits")
        mask=e["batch"]["marker_mask"].bool()
        a=torch.softmax(e["teacher"].masked_fill(~mask,-1e4),-1)
        b=torch.softmax(logits.float().masked_fill(~mask,-1e4),-1)
        av,bv=a.argmax(-1),b.argmax(-1)
        center=(a+b)/2
        js_sum+=float((0.5*(a*(a.clamp_min(1e-9).log()-
                                  center.clamp_min(1e-9).log())).sum(-1)
                      +0.5*(b*(b.clamp_min(1e-9).log()-
                                  center.clamp_min(1e-9).log())).sum(-1)).sum())
        correct+=int((av==bv).sum())
        student_gold+=int((bv==e["gold"]).sum())
        teacher_gold+=int((av==e["gold"]).sum())
        n+=len(av)
    return {"decisions":n,"agreement":correct/max(n,1),
            "student_gold_accuracy":student_gold/max(n,1),
            "teacher_gold_accuracy":teacher_gold/max(n,1),
            "mean_js":js_sum/max(n,1)}
ORIGINAL_GATE=decision_quality(student_eval,GATE_CACHE)
assert ORIGINAL_GATE["agreement"]==1.
print("Original (no compression) DEV-GATE:",ORIGINAL_GATE)


In [ ]:

#@title 5. Native-compatible attention-only operator; chunked SDPA and optional Flex kernel
from tinycenn_lm.laya_lab.core import BaseLayaReplacementAttention
try:
    from torch.nn.attention.flex_attention import (
        flex_attention as _flex_attention,create_block_mask as _create_block_mask)
    FLEX_IMPORTED=True
except ImportError:
    FLEX_IMPORTED=False

if FLEX_IMPORTED and ENABLE_FLEX:
    try:
        # Compile the kernel once; warmup/compilation is excluded from timings.
        _compiled_flex=torch.compile(_flex_attention,dynamic=False)
    except Exception as exc:
        print("Flex compile setup unavailable:",str(exc)[:180])
        _compiled_flex=None
else:
    _compiled_flex=None

def dense_window_sdpa(q,k,v,valid,window):
    t=q.shape[-2]
    if window>=t:
        allowed=valid[:,None,None,:]
    else:
        pos=torch.arange(t,device=q.device)
        radius=(window-1)//2
        near=(pos[:,None]-pos[None,:]).abs()<=radius
        allowed=valid[:,None,None,:]&near[None,None,:,:]
    out=F.scaled_dot_product_attention(q,k,v,attn_mask=allowed,dropout_p=0.0)
    return out*valid[:,None,:,None].to(out.dtype)

def chunked_window_sdpa(q,k,v,valid,window,chunk=64):
    """
    Exact centered local attention without allocating the T-by-T mask.
    Memory is bounded by query chunk × (window+chunk).
    This can still be slower than fused full-attention on short sequences.
    """
    t=q.shape[-2]
    if window>=t:return dense_window_sdpa(q,k,v,valid,window)
    radius=(window-1)//2
    out_chunks=[]
    for start in range(0,t,chunk):
        end=min(start+chunk,t)
        left=max(0,start-radius)
        right=min(t,end+radius)
        qi=torch.arange(start,end,device=q.device)
        ki=torch.arange(left,right,device=q.device)
        near=(qi[:,None]-ki[None,:]).abs()<=radius
        allowed=valid[:,None,None,left:right]&near[None,None,:,:]
        # Query rows with no valid local keys can occur at padded boundaries.
        # Include the diagonal for padded queries, then zero padded outputs.
        empty=~allowed.any(dim=-1,keepdim=True)
        self_key=(qi[:,None]==ki[None,:])[None,None,:,:]
        allowed=allowed|(empty&self_key)
        out=F.scaled_dot_product_attention(
            q[:,:,start:end],k[:,:,left:right],v[:,:,left:right],
            attn_mask=allowed,dropout_p=0.0)
        out_chunks.append(out)
    result=torch.cat(out_chunks,dim=-2)
    return result*valid[:,None,:,None].to(result.dtype)

def flex_window_sdpa(q,k,v,valid,window):
    if _compiled_flex is None:
        raise RuntimeError("Flex kernel unavailable")
    t=q.shape[-2]
    if window>=t:return dense_window_sdpa(q,k,v,valid,window)
    radius=(window-1)//2
    def mask_mod(b,h,q_idx,kv_idx):
        same=(q_idx==kv_idx)
        q_good=valid[b,q_idx]
        k_good=valid[b,kv_idx]
        return (q_good&k_good&((q_idx-kv_idx).abs()<=radius)) | (~q_good&same)
    bm=_create_block_mask(mask_mod,q.shape[0],1,t,t,device=q.device,BLOCK_SIZE=64)
    result=_compiled_flex(q,k,v,block_mask=bm)
    return result*valid[:,None,:,None].to(result.dtype)

class WindowCeNNAttention(BaseLayaReplacementAttention):
    architecture="jev_cenn_single_attention_frozen_ffn_v2"
    def __init__(self,native_attn,window=129,backend="dense"):
        super().__init__(native_attn)
        self.window=int(window)
        self.backend=backend
        self.backend_used=None
        self.flex_failure=None
        self.cenn=CellularMultiscale(self.hidden_size)
        self.cenn_gain=nn.Parameter(torch.tensor(0.0))
        self.use_cenn=True
        # Keep pretrained RoPE/QKV/output projection; train only attention matrices
        # and the zero-initialized optional CeNN path.
        self.Wqkv.requires_grad_(True)
        self.Wo.requires_grad_(True)
        self.out_drop.requires_grad_(False)

    def forward(self,hidden_states,position_embeddings=None,attention_mask=None,**kwargs):
        q,k,v=self.qkv(hidden_states,position_embeddings)
        valid=_valid_tokens(attention_mask,hidden_states)
        if self.backend=="flex" and self.window<q.shape[-2]:
            try:
                y=flex_window_sdpa(q,k,v,valid,self.window)
                self.backend_used="flex"
            except (RuntimeError,NotImplementedError,AssertionError) as exc:
                self.flex_failure=str(exc)[:300]
                y=chunked_window_sdpa(q,k,v,valid,self.window)
                self.backend_used="chunked_fallback"
        elif self.backend=="chunked" and self.window<q.shape[-2]:
            y=chunked_window_sdpa(q,k,v,valid,self.window)
            self.backend_used="chunked"
        else:
            y=dense_window_sdpa(q,k,v,valid,self.window)
            self.backend_used="dense_sdpa"
        if self.use_cenn:
            # Starts exactly zero; max magnitude gain 0.10.
            flattened=y.transpose(1,2).reshape(hidden_states.shape)
            corr=self.cenn(flattened,valid)
            flattened=flattened+0.1*torch.tanh(self.cenn_gain)*corr
            y=flattened.reshape(q.shape[0],q.shape[2],q.shape[1],q.shape[3]).transpose(1,2)
        return self.finish(y,hidden_states)

def make_candidate(layer_index,window):
    # Replacing ONLY encoder.layers[i].attn. FFN, attention norm and residual
    # are the SAME native modules, not replicas of approximated blocks.
    native=teacher.encoder.layers[layer_index].attn
    c=WindowCeNNAttention(native,window=window,backend="dense").to(device).eval()
    return c

def feature_probe(candidate,cache,layer_index):
    # Compare entire native layer output (including unchanged residual/FFN).
    native=student_eval.encoder.layers[layer_index]
    original_attn=native.attn
    native.attn=candidate
    try:
        cs=err=energy=tokens=0.0
        candidate.eval()
        with torch.inference_mode():
            for row in cache:
                e=to_cuda(row)
                # Layer-level direct forward uses cached exact position_ids and
                # a token mask. The attention backend reads 2D valid masks.
                pred=native(e["rollout"],attention_mask=e["valid"],
                            position_ids=e["position_ids"])[0]
                a=pred.float()[e["valid"]]
                b=e["end_target"].float()[e["valid"]]
                check_finite(a,"probe")
                cs+=float(F.cosine_similarity(a,b,dim=-1).sum())
                err+=float((a-b).square().sum())
                energy+=float(b.square().sum())
                tokens+=int(e["valid"].sum())
        return {"cosine":cs/max(tokens,1),"nmse":err/max(energy,1e-8),
                "tokens":int(tokens)}
    finally:
        native.attn=original_attn

def local_loss(pred,target,valid):
    a=pred.float()[valid];b=target.float()[valid]
    nmse=(a-b).square().sum()/b.square().sum().clamp_min(1e-8)
    cos=F.cosine_similarity(a,b,dim=-1).mean()
    return nmse+COSINE_WEIGHT*(1-cos),nmse.detach(),cos.detach()

def verify_reference(layer_index,cache):
    # Faithful SAME-ATTENTION control (512 for native global, 129 for local).
    nat=teacher.encoder.layers[layer_index]
    is_global=layer_index in GLOBAL_LAYERS
    window=512 if is_global else 129
    cand=make_candidate(layer_index,window)
    cand.use_cenn=False
    # Use full-model actual encoder path with native layer, not a standalone
    # direct layer call that changes the upstream attention-mask semantics.
    current=student_eval.encoder.layers[layer_index].attn
    student_eval.encoder.layers[layer_index].attn=cand
    try:
        q=decision_quality(student_eval,GATE_CACHE)
        # For the reference test only, student prefix must be original.
        if layer_index==TOP_LAYER:
            assert q["agreement"] >= 0.995, (
                "Reference attention does not reproduce native teacher. "
                "Stop before training; inspect mask, residual, RoPE.")
        print("Native-attention equivalence layer",layer_index,
              "window",window,"DEV agreement",q["agreement"])
        return q
    finally:
        student_eval.encoder.layers[layer_index].attn=current

assert frozen_fingerprint(student_eval)==ALL_FROZEN_INITIAL
print("Attention-only operator ready; all pretrained FFNs untouched.")


In [ ]:

#@title 6. Fast local distillation with frozen native FFNs (no full-model gradients)
def fit_attention(candidate,layer_index,train_cache,probe_cache,steps):
    # Freeze all of the 28 student blocks; this independent module alone learns.
    candidate.train()
    candidate.backend="dense" # fast/stable training; backend ablation later
    p_att=list(candidate.Wqkv.parameters())+list(candidate.Wo.parameters())
    p_cenn=list(candidate.cenn.parameters())+[candidate.cenn_gain]
    assert all(p.requires_grad for p in p_att+p_cenn)
    optimizer=torch.optim.AdamW([
        {"params":p_att,"lr":LR_ATTENTION},
        {"params":p_cenn,"lr":LR_CENN},
    ],weight_decay=1e-4)
    best_rank=float("inf");best_state=None;best_step=0;history=[]
    indices=list(range(len(train_cache)))
    rng=random.Random(SEED+layer_index*31+candidate.window)
    index=0
    for step in range(1,steps+1):
        if index==0:rng.shuffle(indices)
        row=to_cuda(train_cache[indices[index]])
        index=(index+1)%len(indices)
        optimizer.zero_grad(set_to_none=True)
        # Differentiable forward through only ONE native-shaped layer.
        native=student_eval.encoder.layers[layer_index]
        original_attn=native.attn
        native.attn=candidate
        try:
            out=native(row["rollout"],attention_mask=row["valid"],
                       position_ids=row["position_ids"])[0]
            loss,_,_=local_loss(out,row["end_target"],row["valid"])
            require_finite(loss,"layer attention-only distill")
            loss.backward()
        finally:
            native.attn=original_attn
        torch.nn.utils.clip_grad_norm_(p_att+p_cenn,0.5)
        optimizer.step()
        if step==1 or step%PROBE_EVERY==0 or step==steps:
            candidate.eval()
            probe=feature_probe(candidate,probe_cache,layer_index)
            rank=probe["nmse"]+COSINE_WEIGHT*(1-probe["cosine"])
            if rank<best_rank:
                best_rank=rank;best_step=step
                best_state=cpu_state_snapshot(candidate)
            history.append({"step":step,"probe":probe,"rank":rank})
            print(f"L{layer_index} W{candidate.window} step {step}/{steps} "
                  f"cos={probe['cosine']:.6f} nmse={probe['nmse']:.3g}")
            candidate.train()
    assert best_state is not None
    candidate.load_state_dict(best_state,strict=True)
    candidate.eval().requires_grad_(False)
    assert frozen_fingerprint(student_eval)==ALL_FROZEN_INITIAL
    return {"best_step":best_step,"history":history}

def evaluate_variant(candidate,layer_index,cache,ce_enabled,backend):
    candidate.use_cenn=bool(ce_enabled)
    candidate.backend=backend
    native=student_eval.encoder.layers[layer_index]
    original=native.attn
    before=decision_quality(student_eval,GATE_CACHE)
    native.attn=candidate
    try:
        with torch.inference_mode():
            after=decision_quality(student_eval,GATE_CACHE)
            local=feature_probe(candidate,cache,layer_index)
        verdict=judge_replacement(
            before,after,local_cosine=local["cosine"],local_nmse=local["nmse"],
            min_agreement=TARGET_AGREEMENT,
            max_incremental_agreement_drop=MAX_INCREMENTAL_AGREEMENT_DROP,
            max_gold_drop=MAX_GOLD_DROP,
            min_local_cosine=TARGET_COSINE,max_local_nmse=TARGET_NMSE,
            require_speedup=False)
        return {"layer":layer_index,"window":candidate.window,
                "cenn_enabled":bool(ce_enabled),"backend":backend,
                "backend_used":candidate.backend_used,
                "backend_failure":candidate.flex_failure,
                "before":before,"after":after,"local":local,
                "gate":verdict,"quality_accepted":bool(verdict["accepted"])}
    finally:
        native.attn=original


In [ ]:

#@title 7. Proper paired GPU timings and short/long input selection
@torch.inference_mode()
def timed_forward(model,inp,repeats=BENCH_REPEATS):
    model.eval()
    for _ in range(5):model(**inp)
    torch.cuda.synchronize()
    vals=[]
    for _ in range(repeats):
        a=torch.cuda.Event(enable_timing=True)
        b=torch.cuda.Event(enable_timing=True)
        a.record()
        model(**inp)
        b.record()
        b.synchronize()
        vals.append(float(a.elapsed_time(b)))
    return {"median_ms":float(np.median(vals)),
            "p95_ms":float(np.percentile(vals,95)),
            "min_ms":float(min(vals)),"repeats":repeats}

def matched_speed(candidate,layer_index,inp,repeats=BENCH_REPEATS):
    layer=student_eval.encoder.layers[layer_index]
    original=layer.attn
    try:
        layer.attn=original
        baseline=timed_forward(student_eval,inp,repeats)
        layer.attn=candidate
        alternate=timed_forward(student_eval,inp,repeats)
        # Replay native after alternate to identify large time drift.
        layer.attn=original
        check=timed_forward(student_eval,inp,max(4,repeats//2))
        native_ref=0.5*(baseline["median_ms"]+check["median_ms"])
        return {"native_ms":native_ref,"candidate_ms":alternate["median_ms"],
                "native_a":baseline,"native_b":check,
                "candidate":alternate,"speedup":native_ref/max(alternate["median_ms"],1e-9),
                "backend_used":candidate.backend_used}
    finally:
        layer.attn=original

BENCH_INPUT=GATE_CACHE[0]["batch"]
def actual_length(item):
    return len(item["ids"])

def bench_pools(items):
    return {
        "short": [x for x in items if actual_length(x)<=160],
        "long":  [x for x in items if 384<=actual_length(x)<=MAX_LEN],
    }

def benchmark_lengths(teacher_model,student_model,items):
    pools=bench_pools(items)
    results=[]
    for label,candidates in pools.items():
        if len(candidates)<2:
            results.append({"length_class":label,
              "status":"not_enough_real_examples","eligible":len(candidates)})
            continue
        for size in (1,2):
            chosen=candidates[:size]
            inp=direct_batch(chosen)
            # Native and candidate consume identical tensors, including padding.
            lengths=[actual_length(z) for z in chosen]
            baseline=timed_forward(teacher_model,inp)
            final=timed_forward(student_model,inp)
            results.append({"length_class":label,"batch":size,
                            "lengths":lengths,"padded_length":int(inp["input_ids"].shape[1]),
                            "native":baseline,"final":final,
                            "speedup":baseline["median_ms"]/max(final["median_ms"],1e-9),
                            "status":"measured"})
            print(label,"batch",size,"lengths",lengths,
                  "ratio",round(results[-1]["speedup"],4))
    return results


In [ ]:

#@title 8. Layer 15: test W129/W257/W512, CeNN on/off, actual efficient-kernel parity
L15_TRAIN,L15_PROBE=new_layer_caches(TOP_LAYER)
reference=verify_reference(TOP_LAYER,L15_PROBE)
layer15_rows=[]
all_layer15={}
for window in WINDOWS:
    print("\n=== Layer 15, window",window,"===")
    cand=make_candidate(TOP_LAYER,window)
    fit=fit_attention(cand,TOP_LAYER,L15_TRAIN,L15_PROBE,TRAIN_STEPS)
    configs=[]
    for ce in (False,True):
        # Always validate exact same weights with both CeNN states.
        for backend in ("dense","chunked","flex") if window<MAX_LEN else ("dense",):
            if backend=="flex" and not ENABLE_FLEX:continue
            result=evaluate_variant(cand,TOP_LAYER,L15_PROBE,ce,backend)
            result["training"]=fit
            if result["quality_accepted"]:
                cand.use_cenn=ce;cand.backend=backend
                # Kernel errors/compilation on unsupported GPU are handled by
                # explicit fallback, and the actual backend is recorded.
                result["timing"]=matched_speed(cand,TOP_LAYER,BENCH_INPUT)
                result["backend_used"]=cand.backend_used
                result["backend_failure"]=cand.flex_failure
                if (backend=="flex" and
                    result["backend_used"]!="flex"):
                    result["flex_unsupported_or_failed"]=True
            else:
                result["timing"]=None
            layer15_rows.append(result)
            configs.append(result)
            print(f"Window={window} CeNN={ce} kernel={backend} "
                  f"agreement={result['after']['agreement']:.4f} "
                  f"accepted={result['quality_accepted']} "
                  f"backend_used={result['backend_used']}")
    all_layer15[window]={
        "model":cand,
        "training":fit,
        "evaluations":configs}
    (OUTPUT_DIR/"window_ablation_progress.json").write_text(
        json.dumps(layer15_rows,indent=2,allow_nan=False))
    assert frozen_fingerprint(student_eval)==ALL_FROZEN_INITIAL

# Choose only DEV-passing implementations. Prefer measured >1 speedup; otherwise
# highest speedup with highest fidelity as tie-break. Do NOT optimize on HOLDOUT.
eligible=[v for v in layer15_rows if v["quality_accepted"] and v["timing"]]
if REQUIRE_FASTER_FOR_ACCEPTANCE:
    eligible=[v for v in eligible if v["timing"]["speedup"]>1.0]
eligible.sort(key=lambda r:(
    r["timing"]["speedup"],r["local"]["cosine"],-r["local"]["nmse"]),reverse=True)
L15_SELECTED=eligible[0] if eligible else None
if L15_SELECTED is None:
    print("NO layer15 quality/speed accepted. Native L15 remains.")
    SELECTED_LAYERS=[]
else:
    chosen=all_layer15[L15_SELECTED["window"]]["model"]
    chosen.use_cenn=L15_SELECTED["cenn_enabled"]
    chosen.backend=L15_SELECTED["backend"]
    chosen.eval().requires_grad_(False)
    student_eval.encoder.layers[TOP_LAYER].attn=chosen
    SELECTED_LAYERS=[TOP_LAYER]
    print("Selected:",{"window":L15_SELECTED["window"],
         "cenn":L15_SELECTED["cenn_enabled"],
         "backend":L15_SELECTED["backend"],"measured_speedup":L15_SELECTED["timing"]["speedup"]})
assert frozen_fingerprint(student_eval)==ALL_FROZEN_INITIAL


In [ ]:

#@title 9. Progressive attention-only replacements, preserving every native FFN
PROGRESSIVE_REPORT=[]
if L15_SELECTED is None:
    print("L15 was rejected: no progressive changes applied.")
else:
    for layer_index in PROGRESSIVE_LAYERS:
        print("\n==== PROGRESSIVE layer",layer_index,"====")
        train_cache,probe_cache=new_layer_caches(layer_index)
        candidate=make_candidate(layer_index,PROGRESSIVE_WINDOW)
        fit=fit_attention(candidate,layer_index,train_cache,probe_cache,PROGRESSIVE_STEPS)
        variants=[]
        for ce in (False,True):
            res=evaluate_variant(candidate,layer_index,probe_cache,ce,"dense")
            res["train"]=fit
            if res["quality_accepted"]:
                candidate.use_cenn=ce
                candidate.backend="dense"
                res["timing"]=matched_speed(candidate,layer_index,BENCH_INPUT)
            else: res["timing"]=None
            variants.append(res)
        viable=[v for v in variants if v["quality_accepted"]]
        if REQUIRE_FASTER_FOR_ACCEPTANCE:
            viable=[v for v in viable if v["timing"] and v["timing"]["speedup"]>1.0]
        viable.sort(key=lambda v:(v["timing"]["speedup"] if v["timing"] else -1),reverse=True)
        selected=viable[0] if viable else None
        if selected:
            candidate.use_cenn=selected["cenn_enabled"]
            candidate.backend="dense"
            candidate.eval().requires_grad_(False)
            student_eval.encoder.layers[layer_index].attn=candidate
            SELECTED_LAYERS.append(layer_index)
            print("ACCEPTED L",layer_index,
                  "cumulative agreement",selected["after"]["agreement"])
        else:
            print("REJECTED L",layer_index,"; kept original attention.")
        PROGRESSIVE_REPORT.append({
            "layer":layer_index,"accepted":bool(selected),
            "selected":selected,"variants":variants})
        assert frozen_fingerprint(student_eval)==ALL_FROZEN_INITIAL
        (OUTPUT_DIR/"progressive_results.json").write_text(
            json.dumps(PROGRESSIVE_REPORT,indent=2,allow_nan=False))
print("Only attention modules replaced:",SELECTED_LAYERS)
print("All 28 original FFNs and normalization modules intact.")


In [ ]:

#@title 10. ONE-TIME independent holdout, real short/long GPU speed, publication artifacts
# None of the following results are used to accept/reject/select configurations.
holdout_items=make_items_from_rows(holdout_rows,SEED+999,False)
assert len(holdout_items)>=HOLDOUT_DECISIONS
HOLDOUT_ITEMS=holdout_items[:HOLDOUT_DECISIONS]
HOLDOUT_CACHE=cache_teacher_predictions(HOLDOUT_ITEMS)
HOLDOUT=decision_quality(student_eval,HOLDOUT_CACHE)
FINAL_DEV=decision_quality(student_eval,GATE_CACHE)
final_speed=benchmark_lengths(teacher,student_eval,
                              GATE_ITEMS+probe_items)
baseline_param_count=sum(x.numel() for x in teacher.parameters())
final_param_count=sum(x.numel() for x in student_eval.parameters())
change_in_params=final_param_count-baseline_param_count
assert frozen_fingerprint(student_eval)==ALL_FROZEN_INITIAL
selected_config={
    "layer":TOP_LAYER,"result":L15_SELECTED,
    "selected_layers":SELECTED_LAYERS,"progressive":PROGRESSIVE_REPORT}
report={
    "architecture":"attention_submodule_only_ffn_preserved_L15_v2",
    "source_model":SOURCE_MODEL,"seed":SEED,"torch_version":torch.__version__,
    "gpu":torch.cuda.get_device_name(0),"original_encoder_layers":28,
    "target":15,"window_candidates":list(WINDOWS),
    "windows_and_cenn_ablation":layer15_rows,
    "selected_layer15":L15_SELECTED,
    "progressive":PROGRESSIVE_REPORT,
    "accepted_attention_layers":SELECTED_LAYERS,
    "native_ffn_norms_preserved":True,
    "frozen_ffn_norm_sha256":ALL_FROZEN_INITIAL,
    "dev_final":FINAL_DEV,
    "sealed_holdout":HOLDOUT,
    "speed_short_long":final_speed,
    "native_parameter_count":baseline_param_count,
    "final_parameter_count":final_param_count,
    "added_parameter_count":change_in_params,
    "model_parameter_reduction_claimed":False,
    "training_steps_primary":TRAIN_STEPS,
    "training_steps_progressive":PROGRESSIVE_STEPS,
    "dataset_partitions":split_info,
    "limitations":[
      "The heldout split is case-isolated but not a new dataset.",
      "Original model weights may already have seen similar training distributions.",
      "CeNN on/off is an ablation of an attention-only approximation, not isolated proof that CeNN replaces full attention.",
      "Window 512 is the exact full-attention CONTROL at max sequence length 512, not an asymmetric centered window.",
      "Flex backend may fall back to chunked on some Colab GPUs; see per-row actual backend.",
      "Chunked SDPA reduces mask workspace but can be slower on small inputs.",
      "All native FFNs are retained: this experiment may add parameters rather than compress the model.",
      "Short/long benchmarks report not_enough_real_examples rather than invent long sequences.",
      "GPU event times are runtime-, dtype- and batch-specific; reproducibility requires multiple hardware trials."
    ]}
(OUTPUT_DIR/"layer15_complete_report.json").write_text(
    json.dumps(report,indent=2,allow_nan=False))
import csv
with (OUTPUT_DIR/"layer15_ablation.csv").open("w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=[
        "layer","window","cenn_enabled","backend","backend_used",
        "quality_accepted","agreement","gold_accuracy","cosine","nmse",
        "speedup","native_ms","candidate_ms"])
    w.writeheader()
    for v in layer15_rows:
        speed=v["timing"] or {}
        w.writerow({
            "layer":v["layer"],"window":v["window"],
            "cenn_enabled":v["cenn_enabled"],"backend":v["backend"],
            "backend_used":v["backend_used"],
            "quality_accepted":v["quality_accepted"],
            "agreement":v["after"]["agreement"],
            "gold_accuracy":v["after"]["student_gold_accuracy"],
            "cosine":v["local"]["cosine"],"nmse":v["local"]["nmse"],
            "speedup":speed.get("speedup"),
            "native_ms":speed.get("native_ms"),
            "candidate_ms":speed.get("candidate_ms")})
for i in SELECTED_LAYERS:
    attn=student_eval.encoder.layers[i].attn
    torch.save({"source_model":SOURCE_MODEL,"layer":i,
        "architecture":attn.architecture,"window":attn.window,
        "cenn_enabled":attn.use_cenn,
        "backend":attn.backend,"state":cpu_state_snapshot(attn)},
        OUTPUT_DIR/f"accepted_L{i:02d}_attention.pt")
print(json.dumps({
 "accepted_attention_layers":SELECTED_LAYERS,
 "selected_window":L15_SELECTED["window"] if L15_SELECTED else None,
 "dev_agreement":FINAL_DEV["agreement"],
 "holdout_agreement":HOLDOUT["agreement"],
 "holdout_gold":HOLDOUT["student_gold_accuracy"],
 "original_params":baseline_param_count,
 "final_params":final_param_count,
 "output":str(OUTPUT_DIR)},indent=2))
import zipfile
archive=OUTPUT_DIR/"layer15_full_results.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.glob("*"):
        if p!=archive and p.is_file():z.write(p,p.name)
print("Download results:",archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    pass
